<div dir="rtl">

# ۰۲ — کلاینت پایتونی با SDK رسمی و MCP Inspector

در نوت‌بوک قبل همه‌ی پیام‌ها را دستی ساختیم. حالا همان کارها را با کلاینت **SDK رسمی MCP** انجام می‌دهیم و بعد با ابزار گرافیکی **MCP Inspector** سرور را تست می‌کنیم.

</div>

In [1]:
from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY (and other settings) from the .env file

from mcp_helpers import setup_notebook, start_server, stop_server, parse_sse, show_json
setup_notebook()

Notebook ready | Python 3.12.12 | win32


<div dir="rtl">

همان سرور نوت‌بوک قبل را می‌سازیم. اگر فایل را از قبل دارید، این سلول فقط آن را بازنویسی می‌کند.

</div>

In [2]:
import os
os.makedirs("servers", exist_ok=True)

In [3]:
%%writefile servers/demo_server.py
from mcp.server.fastmcp import FastMCP

# The name is what clients see in "serverInfo"
mcp = FastMCP("Demo Server", port=8000)


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b


@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


if __name__ == "__main__":
    # Streamable HTTP: the server listens on http://127.0.0.1:8000/mcp
    mcp.run(transport="streamable-http")


Overwriting servers/demo_server.py


In [4]:
server = start_server("servers/demo_server.py", port=8000)

Server started: servers/demo_server.py   (port 8000, pid 22720, log: logs\demo_server_8000.log)


<div dir="rtl">

## دو لایه‌ی کلاینت

اتصال در SDK از دو لایه تشکیل شده است:

۱. **لایه‌ی انتقال (transport):** `streamable_http_client(URL)` ارتباط HTTP را برقرار می‌کند و دو «جریان» برمی‌گرداند: `read` برای دریافت پیام از سرور و `write` برای ارسال پیام به سرور. تابع سوم، `get_session_id`، شناسه‌ی session را برمی‌گرداند.

۲. **لایه‌ی session:** `ClientSession(read, write)` پیام‌های JSON-RPC را می‌سازد، `id` می‌دهد و پاسخ‌ها را با درخواست‌ها تطبیق می‌دهد. همه‌ی کارهای دستی نوت‌بوک قبل را این کلاس انجام می‌دهد.

این جدا بودن مهم است: اگر فردا بخواهید به جای HTTP از stdio استفاده کنید (نوت‌بوک بعد)، فقط لایه‌ی اول عوض می‌شود. `ClientSession` و همه‌ی کدهای بعدی همان می‌مانند.

</div>

In [5]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

URL = "http://127.0.0.1:8000/mcp"

async with streamable_http_client(URL) as (read, write, get_session_id):
    async with ClientSession(read, write) as session:
        print("Session ID before initialize:", get_session_id())

        init = await session.initialize()  # initialize + notifications/initialized

        print("Session ID after initialize: ", get_session_id())
        print("Protocol version:", init.protocolVersion)
        print("Server:", init.serverInfo.name)
        print("Capabilities:", init.capabilities.model_dump(exclude_none=True))

Session ID before initialize: None
Session ID after initialize:  824e9fab76d34c98b45eb5ceabe02a39
Protocol version: 2025-11-25
Server: Demo Server
Capabilities: {'experimental': {}, 'prompts': {'listChanged': False}, 'resources': {'subscribe': False, 'listChanged': False}, 'tools': {'listChanged': False}}


<div dir="rtl">

فقط با یک خط `await session.initialize()` هر دو پیام `initialize` و `notifications/initialized` فرستاده شدند و شناسه‌ی session هم خودکار ذخیره شد. وقتی از بلوک `async with` خارج می‌شویم، SDK درخواست `DELETE` را هم خودش می‌فرستد.

> **نکته:** در Jupyter می‌توانیم مستقیم از `await` استفاده کنیم. در یک فایل `.py` معمولی، این کد باید داخل یک تابع `async` باشد و با `asyncio.run(main())` اجرا شود (پایین نوت‌بوک یک نمونه آمده است).

برای اینکه در هر سلول این دو لایه را تکرار نکنیم، یک تابع کمکی می‌سازیم:

</div>

In [6]:
from contextlib import asynccontextmanager


@asynccontextmanager
async def connect(url=URL):
    async with streamable_http_client(url) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            yield session

<div dir="rtl">

چرا `yield` و نه `return`؟ تابع `connect` باید **وسط کار مکث کند**: session را به `async with connect() as session` بدهد، صبر کند تا کار ما در آن بلوک تمام شود، و بعد برگردد و از دو `async with` بیرون برود (همان `DELETE`). `return` این مکث را ندارد؛ از هر دو `async with` بیرون می‌رود و یک session **بسته‌شده** برمی‌گرداند که اولین `call_tool` روی آن خطای `ClosedResourceError` می‌دهد.

دکوریتور `@asynccontextmanager` تابع را از جای `yield` دو نیم می‌کند: قبل از آن «ورود» است، مقدار yield‌شده همان `as session` است، و بعد از آن «خروج». اگر داخل بلوک خطایی رخ دهد، باز هم اتصال مرتب بسته می‌شود.

</div>

<div dir="rtl">

## لیست ابزارها

</div>

In [7]:
async with connect() as session:
    result = await session.list_tools()

for tool in result.tools:
    print(f"{tool.name}: {tool.description}")
    show_json(tool.inputSchema)
    print()

add: Add two integers and return the sum.
{
  "properties": {
    "a": {
      "title": "A",
      "type": "integer"
    },
    "b": {
      "title": "B",
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ],
  "title": "addArguments",
  "type": "object"
}

divide: Divide a by b.
{
  "properties": {
    "a": {
      "title": "A",
      "type": "number"
    },
    "b": {
      "title": "B",
      "type": "number"
    }
  },
  "required": [
    "a",
    "b"
  ],
  "title": "divideArguments",
  "type": "object"
}



<div dir="rtl">

خروجی همان `tools/list` نوت‌بوک قبل است، ولی حالا به‌صورت شیء پایتون (`Tool`) با فیلدهای `name`، `description`، `inputSchema` و `outputSchema`.

## فراخوانی ابزار

`call_tool` یک شیء `CallToolResult` برمی‌گرداند:

</div>

In [8]:
async with connect() as session:
    result = await session.call_tool("add", {"a": 21, "b": 21})

print("isError:          ", result.isError)
print("content (text):   ", result.content[0].text)
print("structuredContent:", result.structuredContent)

isError:           False
content (text):    42
structuredContent: {'result': 42}


<div dir="rtl">

## خطاها

یادآوری از نوت‌بوک قبل: اگر ابزار هنگام اجرا خطا بدهد، `call_tool` **exception نمی‌دهد**. یک نتیجه‌ی عادی با `isError=True` برمی‌گرداند تا متن خطا به مدل زبانی برسد. حتی صدا زدن ابزاری که وجود ندارد هم همین‌طور است:

</div>

In [9]:
async with connect() as session:
    for name, args in [("divide", {"a": 1, "b": 0}), ("multiply", {"a": 2, "b": 3})]:
        result = await session.call_tool(name, args)
        print(f"{name:10s} isError={result.isError}  ->  {result.content[0].text}")

divide     isError=True  ->  Error executing tool divide: float division by zero
multiply   isError=True  ->  Unknown tool: multiply


<div dir="rtl">

## مقایسه با نوت‌بوک ۰۱

| کار دستی در ۰۱ | با SDK |
|---|---|
| ساخت پیام `initialize`، خواندن هدر session، ارسال `notifications/initialized` | `await session.initialize()` |
| اضافه کردن `Mcp-Session-Id` و `MCP-Protocol-Version` به همه‌ی درخواست‌ها | خودکار |
| شماره‌گذاری `id` و تطبیق پاسخ با درخواست | خودکار |
| بیرون کشیدن JSON از قالب SSE | خودکار |
| `tools/list` و `tools/call` | `list_tools()` و `call_tool()` |
| `DELETE` برای پایان session | خروج از `async with` |

## همین کلاینت در یک فایل پایتون معمولی

خارج از Jupyter، کد باید داخل یک تابع `async` باشد:

</div>

In [10]:
%%writefile servers/demo_client.py
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def main():
    async with streamable_http_client("http://127.0.0.1:8000/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("add", {"a": 21, "b": 21})
            print("Server result:", result.content[0].text)


if __name__ == "__main__":
    asyncio.run(main())


Overwriting servers/demo_client.py


<div dir="rtl">

این فایل را در یک ترمینال دیگر اجرا کنید، در حالی که سرور روشن است:

- ویندوز: `python servers\demo_client.py`
- لینوکس: `python3 servers/demo_client.py`

یا از داخل نوت‌بوک:

</div>

In [11]:
import subprocess, sys
print(subprocess.run([sys.executable, "servers/demo_client.py"], capture_output=True, text=True).stdout)

Server result: 42



<div dir="rtl">

## تست سرور با MCP Inspector

**MCP Inspector** یک ابزار گرافیکی رسمی برای تست سرورهای MCP است: ابزارها را لیست می‌کند، اجازه می‌دهد آن‌ها را با ورودی دلخواه صدا بزنید و همه‌ی پیام‌های JSON-RPC را نشان می‌دهد.

پیش‌نیاز آن Node.js است (نوت‌بوک ۰۰). نسخه‌ی فعلی Inspector (2.x) نسخه‌ی تازه‌ای از Node می‌خواهد: `22.23.4` به بالا در خط ۲۲، یا `24.22.1` به بالا، یا `26.12.1` به بالا. نسخه‌ی خودتان را با `node --version` ببینید؛ اگر قدیمی‌تر است، Node LTS را دوباره نصب کنید.

**اتصال به سرور HTTP که الان روشن است.** در یک ترمینال جدید بزنید:

</div>

```bash
npx @modelcontextprotocol/inspector
```

<div dir="rtl">

۱. آدرسی مثل `http://127.0.0.1:6274?MCP_INSPECTOR_API_TOKEN=...` چاپ می‌شود و مرورگر باز می‌شود. اگر پرسید `Need to install the following packages`، کلید `y` را بزنید.

۲. صفحه‌ی **Servers** باز می‌شود که چند سرور نمونه دارد. روی **Add Servers** و بعد **+ Add manually** بزنید.

۳. فرم را پر کنید و **Add** را بزنید:
- **Server ID:** `demo`
- **Transport:** `streamable-http`
- **URL:** `http://127.0.0.1:8000/mcp`

۴. کلید کنار کارت `demo` را روشن کنید تا وضعیت **Connected** شود. روی کارت، نسخه‌ی پروتکل توافق‌شده (`MCP 2025-11-25`) هم نوشته می‌شود.

۵. از تب **Tools** بالای صفحه، ابزار `add` را انتخاب کنید، `A` و `B` را پر کنید و **Execute Tool** را بزنید. نتیجه در بخش **Results** نمایش داده می‌شود.

۶. پنل **Messages** سمت راست همان پیام‌هایی را نشان می‌دهد که در نوت‌بوک ۰۱ دستی ساختیم: `initialize`، `notifications/initialized`، `tools/list` و `tools/call`. برچسب **LEGACY** کنار آن یعنی پروتکل session‌دار (پیش از ۲۰۲۶-۰۷-۲۸)، همان که در این دوره استفاده می‌کنیم.

برای بستن Inspector در ترمینال `Ctrl+C` را بزنید. اگر هنگام اجرا پیام `PORT IS IN USE` دیدید، یک Inspector قبلی هنوز باز است؛ اول آن را ببندید.

**راه کوتاه‌تر:** آدرس سرور را از خط فرمان بدهید. سرور آماده در فهرست می‌آید و فقط کافی است کلید اتصالش را روشن کنید (در این حالت فهرست سرورها ذخیره نمی‌شود):

</div>

```bash
npx @modelcontextprotocol/inspector --transport http --server-url http://127.0.0.1:8000/mcp
```

<div dir="rtl">

**بدون مرورگر (حالت CLI).** خروجی مستقیم در ترمینال چاپ می‌شود. آدرسی که به `/mcp` ختم شود خودکار HTTP تشخیص داده می‌شود:

</div>

```bash
npx @modelcontextprotocol/inspector --cli http://127.0.0.1:8000/mcp --method tools/list
npx @modelcontextprotocol/inspector --cli http://127.0.0.1:8000/mcp --method tools/call --tool-name add --tool-arg a=2 b=3
```

<div dir="rtl">

**اتصال به یک سرور stdio.** در این حالت Inspector خودش سرور را اجرا می‌کند (نوت‌بوک بعد). هر چه بعد از نام Inspector بیاید **دستور اجرای سرور** است. پس `stdio` در آخر دستور گزینه‌ی Inspector نیست؛ آرگومانی است که به خود `transport_server.py` می‌رسد تا سرور در حالت stdio بالا بیاید. دستور روی هر دو سیستم‌عامل یکی است:

</div>

```bash
npx @modelcontextprotocol/inspector python servers/transport_server.py stdio
```

<div dir="rtl">

در پایان سرور را خاموش می‌کنیم:

</div>

In [ ]:
stop_server(server)